In [10]:
import numpy as np
import pandas as pd
import scipy.io
from scipy.ndimage import uniform_filter1d
import os
import matplotlib.pyplot as plt
from resources.calculations import smooth, get_nodes, get_struts, get_ductileData, get_fractureData

In [2]:
### GLOBAL PATH INPUTS

pData = 'data/'

pAl          = pData + 'Al/'
pAK          = pAl + 'AK/'
pUTdisNodes  = pAK + 'Ductile-disNodes-FCC-12X16/'
pUTdisStruts = pAK + 'Ductile-disStruts-FCC-12X16/'
pFTdisNodes  = pAK + 'Fracture-disNodes/'

pTi    = pData + 'Ti/'
pTiTri = pTi + 'tri-new/'
pTiKag = pTi + 'kagome/'
pTiFCC = pTi + 'FCC/'
pTiHex = pTi + 'hex/'

In [3]:
runMATLAB = False
pMATLAB   = pUTdisNodes

runINOUT  = True
pINOUT    = pTiTri

# MATLAB to CSV Data File Conversion

In [4]:
### INPUT

mode = 'Ductile'
dis = 'disNodes'

MATin  = pMATLAB + 'raw/' + 'inputData-20-frame-0.mat'
MATout = pMATLAB + 'raw/' + 'outputStressStrain-20-1-1500.mat'

CSVin  = pMATLAB + f'{mode}-{dis}-IN.csv'
CSVout = pMATLAB + f'{mode}-{dis}-OUT.csv'
append = False

In [5]:
def mat_to_csv(MATin, MATout, CSVin, CSVout, append=False):
    input_mat = scipy.io.loadmat(MATin)
    in_data = pd.DataFrame(list(input_mat.values())[3])
    in_data = in_data.rename(columns={i:str(i) for i in range(len(in_data.columns))})
    if append:
        in_header = pd.read_csv(CSVin, index_col=0)
        in_header = in_header.rename(columns={i:str(i) for i in range(len(in_header.columns))})
        in_data = in_data + in_header.iloc[0].values
        in_data = pd.concat([in_header, in_data], ignore_index=True)
    in_data.to_csv(CSVin)

    output_mat = scipy.io.loadmat(MATout)
    out_data = pd.DataFrame(list(output_mat.values())[3])
    out_data = out_data.rename(columns={i:str(i) for i in range(len(out_data.columns))})
    if append:
        out_header = pd.read_csv(CSVout, index_col=0)
        out_header = out_header.drop(out_header.columns[201:], axis=1)
        out_header = out_header.rename(columns={i:str(i) for i in range(len(out_header.columns))})
        out_data = pd.concat([out_header, out_data], ignore_index=True)
    out_data.to_csv(CSVout)

In [6]:
if runMATLAB:
    mat_to_csv(MATin, MATout, CSVin, CSVout, append=append)

# Create Input and Output CSV from Raw .csv Files

### Inputs

In [7]:
def create_inputCSV(directory):
    duct_disNodes = []
    duct_disStruts = []
    frac_disNodes = []
    frac_disStruts = []
    
    pathRaw = directory + 'transfer/'
    for ffile in os.listdir(pathRaw):
        num = int(ffile.split("-")[-1][:-4])
        if ('per' in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'IN-n' in ffile):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + ffile)
            duct_disNodes.insert(0, np.insert(nodes_df.to_numpy().flatten(), 0, 0))
        elif ('per' in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'IN-s' in ffile):
            thicks = get_struts(pathRaw + '/' + ffile)
            duct_disStruts.insert(0, np.insert(thicks, 0, 0))
        elif ('per' in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'IN-n' in ffile):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + ffile)
            frac_disNodes.insert(0, np.insert(nodes_df.to_numpy().flatten(), 0, 0))
        elif ('per' in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'IN-s' in ffile):
            thicks = get_struts(pathRaw + '/' + ffile)
            frac_disStruts.insert(0, np.insert(thicks, 0, 0))
        elif ("disNodes" in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'IN-' in ffile):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
            duct_disNodes.append(np.insert(nodes_df.to_numpy().flatten(), 0, num))
        elif ("disStruts" in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'IN-' in ffile):
            thicks = get_struts(pathRaw + '/' + ffile)
            duct_disStruts.append(np.insert(thicks, 0, num))
        elif ("disNodes" in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'IN-' in ffile):
            nodes, nodesCoords, nodes_df = get_nodes(pathRaw + '/' + ffile)
            frac_disNodes.append(np.insert(nodes_df.to_numpy().flatten(), 0, num))
        elif ("disStruts" in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'IN-' in ffile):
            thicks = get_struts(pathRaw + '/' + ffile)
            frac_disStruts.append(np.insert(thicks, 0, num))
    
    UTdisNodesIN_df = pd.DataFrame(duct_disNodes)
    UTdisStrutsIN_df = pd.DataFrame(duct_disStruts)
    FTdisNodesIN_df = pd.DataFrame(frac_disNodes)
    FTdisStrutsIN_df = pd.DataFrame(frac_disStruts)
    
    UTdisNodesIN_df.to_csv(directory + '/Ductile-disNodes-IN.csv')
    UTdisStrutsIN_df.to_csv(directory + '/Ductile-disStruts-IN.csv')
    FTdisNodesIN_df.to_csv(directory + '/Fracture-disNodes-IN.csv')
    FTdisStrutsIN_df.to_csv(directory + '/Fracture-disStruts-IN.csv')
    
    return UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df

### Outputs

In [8]:
def create_outputCSV(directory):
    duct_disNodes = []
    duct_disStruts = []
    frac_disNodes = []
    frac_disStruts = []
    
    pathRaw = directory + 'transfer/'
    for ffile in os.listdir(pathRaw):
        num = int(ffile.split("-")[-1][:-4])
        if ('per' in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'OUT-' in ffile):
            output_df = get_ductileData(pathRaw + ffile, crit=0.4)
            duct_disNodes.insert(0, np.insert(output_df.x.tolist(), 0, 0))
            duct_disStruts.insert(0, np.insert(output_df.x.tolist(), 0, 0))
            duct_disNodes.insert(1, np.insert(output_df.y_sm.tolist(), 0, 0))
            duct_disStruts.insert(1, np.insert(output_df.y_sm.tolist(), 0, 0))
        elif ('per' in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'OUT-' in ffile):
            output_df = get_fractureData(pathRaw + ffile)
            frac_disNodes.insert(0, np.insert(output_df.x.tolist(), 0, 0))
            frac_disStruts.insert(0, np.insert(output_df.x.tolist(), 0, 0))
            frac_disNodes.insert(1, np.insert(output_df.y_sm.tolist(), 0, 0))
            frac_disStruts.insert(1, np.insert(output_df.y_sm.tolist(), 0, 0))
        elif ("disNodes" in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'OUT-' in ffile):
            output_df = get_ductileData(pathRaw + ffile, crit=0.4)
            duct_disNodes.append(np.insert(output_df.y_sm.tolist(), 0, num))
        elif ('disStruts' in ffile and ffile.endswith('.csv') and 'Ductile' in ffile and 'OUT-' in ffile):
            output_df = get_ductileData(pathRaw + ffile, crit=0.4)
            duct_disStruts.append(np.insert(output_df.y_sm.tolist(), 0, num))
        elif ("disNodes" in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'OUT-' in ffile):
            output_df = get_fractureData(pathRaw + ffile)
            frac_disNodes.append(np.insert(output_df.y_sm.tolist(), 0, num))
        elif ('disStruts' in ffile and ffile.endswith('.csv') and 'Fracture' in ffile and 'OUT-' in ffile):
            output_df = get_fractureData(pathRaw + ffile)
            frac_disNodes.append(np.insert(output_df.y_sm.tolist(), 0, num))
    
    UTdisNodesOUT_df = pd.DataFrame(duct_disNodes, index=[int(i[0]) for i in duct_disNodes])
    UTdisStrutsOUT_df = pd.DataFrame(duct_disStruts, index=[int(i[0]) for i in duct_disStruts])
    FTdisNodesOUT_df = pd.DataFrame(frac_disNodes, index=[int(i[0]) for i in frac_disNodes])
    FTdisStrutsOUT_df = pd.DataFrame(frac_disStruts, index=[int(i[0]) for i in frac_disStruts])
    
    UTdisNodes_drop = [i for i,j in zip([int(i[0]) for i in duct_disNodes], UTdisNodesOUT_df.isnull().any(axis=1)) if j == True]
    UTdisNodes_drop = UTdisNodes_drop + [i for i,j in zip([int(i[0]) for i in duct_disNodes], [int(i[1]) for i in duct_disNodes]) if j == 0.0]
    UTdisNodesOUT_df = UTdisNodesOUT_df.drop(UTdisNodes_drop, axis=0).drop(0, axis=1)
    UTdisNodesOUT_df.columns = range(UTdisNodesOUT_df.columns.size)
    UTdisNodesIN_df = pd.read_csv(directory + 'Ductile-disNodes-IN.csv', index_col=0)
    UTdisNodesIN_df.index = [int(i[0]) for i in duct_disNodes][1:]
    UTdisNodesIN_df = UTdisNodesIN_df.drop(UTdisNodes_drop, axis=0).drop('0', axis=1)
    UTdisNodesIN_df.columns = range(UTdisNodesIN_df.columns.size)
    UTdisNodesOUT_df.to_csv(directory + 'Ductile-disNodes-OUT.csv')
    UTdisNodesIN_df.to_csv(directory + 'Ductile-disNodes-IN.csv')
    
    UTdisStruts_drop = [i for i,j in zip([int(i[0]) for i in duct_disStruts], UTdisStrutsOUT_df.isnull().any(axis=1)) if j == True]
    UTdisStruts_drop = UTdisStruts_drop + [i for i,j in zip([int(i[0]) for i in duct_disStruts], [int(i[1]) for i in duct_disStruts]) if j == 0.0]
    UTdisStrutsOUT_df = UTdisStrutsOUT_df.drop(UTdisStruts_drop, axis=0).drop(0, axis=1)
    UTdisStrutsOUT_df.columns = range(UTdisStrutsOUT_df.columns.size)
    UTdisStrutsIN_df = pd.read_csv(directory + 'Ductile-disStruts-IN.csv', index_col=0)
    UTdisStrutsIN_df.index = [int(i[0]) for i in duct_disStruts][1:]
    UTdisStrutsIN_df = UTdisStrutsIN_df.drop(UTdisStruts_drop, axis=0).drop('0', axis=1)
    UTdisStrutsIN_df.columns = range(UTdisStrutsIN_df.columns.size)
    UTdisStrutsOUT_df.to_csv(directory + 'Ductile-disStruts-OUT.csv')
    UTdisStrutsIN_df.to_csv(directory + 'Ductile-disStruts-IN.csv')
    
    FTdisNodes_drop = [i for i,j in zip([int(i[0]) for i in frac_disNodes], FTdisNodesOUT_df.isnull().any(axis=1)) if j == True]
    FTdisNodes_drop = FTdisNodes_drop + [i for i,j in zip([int(i[0]) for i in frac_disNodes], [int(i[1]) for i in frac_disNodes]) if j == 0.0]
    FTdisNodesOUT_df = FTdisNodesOUT_df.drop(FTdisNodes_drop, axis=0).drop(0, axis=1)
    FTdisNodesOUT_df.columns = range(FTdisNodesOUT_df.columns.size)
    FTdisNodesIN_df = pd.read_csv(directory + 'Fracture-disNodes-IN.csv', index_col=0)
    FTdisNodesIN_df.index = [int(i[0]) for i in frac_disNodes][1:]
    FTdisNodesIN_df = FTdisNodesIN_df.drop(FTdisNodes_drop, axis=0).drop('0', axis=1)
    FTdisNodesIN_df.columns = range(FTdisNodesIN_df.columns.size)
    FTdisNodesOUT_df.to_csv(directory + 'Fracture-disNodes-OUT.csv')
    FTdisNodesIN_df.to_csv(directory + 'Fracture-disNodes-IN.csv')
    
    FTdisStruts_drop = [i for i,j in zip([int(i[0]) for i in frac_disStruts], FTdisStrutsOUT_df.isnull().any(axis=1)) if j == True]
    FTdisStruts_drop = FTdisStruts_drop + [i for i,j in zip([int(i[0]) for i in frac_disStruts], [int(i[1]) for i in frac_disStruts]) if j == 0.0]
    FTdisStrutsOUT_df = FTdisStrutsOUT_df.drop(FTdisStruts_drop, axis=0).drop(0, axis=1)
    FTdisStrutsOUT_df.columns = range(FTdisStrutsOUT_df.columns.size)
    FTdisStrutsIN_df = pd.read_csv(directory + 'Fracture-disStruts-IN.csv', index_col=0)
    FTdisStrutsIN_df.index = [int(i[0]) for i in frac_disStruts][1:]
    FTdisStrutsIN_df = FTdisStrutsIN_df.drop(FTdisStruts_drop, axis=0).drop('0', axis=1)
    FTdisStrutsIN_df.columns = range(FTdisStrutsIN_df.columns.size)
    FTdisStrutsOUT_df.to_csv(directory + 'Fracture-disStruts-OUT.csv')
    FTdisStrutsIN_df.to_csv(directory + 'Fracture-disStruts-IN.csv')
    
    return UTdisNodesOUT_df, UTdisStrutsOUT_df, FTdisNodesOUT_df, FTdisStrutsOUT_df, UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df

In [9]:
if runINOUT:
    UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df = create_inputCSV(pINOUT)
    UTdisNodesOUT_df, UTdisStrutsOUT_df, FTdisNodesOUT_df, FTdisStrutsOUT_df, UTdisNodesIN_df, UTdisStrutsIN_df, FTdisNodesIN_df, FTdisStrutsIN_df = create_outputCSV(pINOUT)

In [12]:
#get_FToutputCurve(pTiFCC + "transfer/OUT-Fracture-FCC-30-disNodes-27.csv")
#UTdisNodesOUT_df#.iloc[1][1:].tolist())